In [2]:
import os, numpy as np, pandas as pd, torch, gc, nibabel as nib
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import jaccard_score, accuracy_score
from monai.networks.nets import UNet, SwinUNETR, BasicUNet
from monai.losses import DiceCELoss

# --- CONFIGURATION ---
DATA_ROOT = r"C:\Users\EGlaciers\Desktop\Dataset\BraTS2020_TrainingData\MICCAI_BraTS2020_TrainingData"
SAVE_DIR = r"C:\Users\EGlaciers\Desktop\Project NCKH\BrainMRI\Convert3D"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 1
ACCUM_STEPS = 4       # Virtual Batch Size = 4
EPOCHS = 30         # Đề xuất 100 để Dice đạt đỉnh
LR = 1e-4

SLICE_RANGE = (50, 114) # 64 lát cắt chuẩn
CROP_SIZE = (40, 200)  # 160x160 trung tâm

In [ ]:
def run_preprocessing():
    os.makedirs(SAVE_DIR, exist_ok=True)
    all_p = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
    patient_list = [p for p in all_p if "355" not in p] # Bỏ qua bệnh nhân lỗi 355

    for p_id in tqdm(patient_list, desc="🔄 Preprocessing"):
        img_p = os.path.join(SAVE_DIR, f"{p_id}_img.npy")
        msk_p = os.path.join(SAVE_DIR, f"{p_id}_mask.npy")

        if os.path.exists(img_p) and os.path.exists(msk_p): continue

        try:
            p_folder = os.path.join(DATA_ROOT, p_id)
            v4d = []
            for mod in ['t1', 't1ce', 't2', 'flair']:
                data = nib.load(os.path.join(p_folder, f"{p_id}_{mod}.nii")).get_fdata()
                crop = data[CROP_SIZE[0]:CROP_SIZE[1], CROP_SIZE[0]:CROP_SIZE[1], SLICE_RANGE[0]:SLICE_RANGE[1]]
                # Normalization
                nz = crop > 0
                if nz.sum() > 0:
                    p1, p99 = np.percentile(crop[nz], [1, 99])
                    crop = np.clip(crop, p1, p99)
                    crop = (crop - p1) / (p99 - p1 + 1e-8)
                v4d.append(crop.astype(np.float32))

            mask = nib.load(os.path.join(p_folder, f"{p_id}_seg.nii")).get_fdata()
            mask_c = mask[CROP_SIZE[0]:CROP_SIZE[1], CROP_SIZE[0]:CROP_SIZE[1], SLICE_RANGE[0]:SLICE_RANGE[1]]
            mask_c[mask_c == 4] = 3

            np.save(img_p, np.stack(v4d, axis=0))
            np.save(msk_p, mask_c.astype(np.uint8))
        except Exception as e:
            print(f"❌ Error at {p_id}: {e}")

# Chạy một lần duy nhất
# run_preprocessing()

In [10]:
class BraTS3DDataset(Dataset):
    def __init__(self, p_ids): self.p_ids = p_ids
    def __len__(self): return len(self.p_ids)
    def __getitem__(self, idx):
        p_id = self.p_ids[idx]
        x = np.load(os.path.join(SAVE_DIR, f"{p_id}_img.npy"))
        y = np.load(os.path.join(SAVE_DIR, f"{p_id}_mask.npy"))
        # (C, H, W, D) -> (C, D, H, W) cho chuẩn MONAI/PyTorch 3D
        return torch.from_numpy(x).permute(0, 3, 1, 2).float(), \
               torch.from_numpy(y).permute(2, 0, 1).long()

class DeepLabV3Plus3D(nn.Module):
    def __init__(self):
        super().__init__()
        # Backbone 6 tầng để tránh lỗi size
        self.backbone = BasicUNet(spatial_dims=3, in_channels=4, out_channels=32,
                                 features=(32, 32, 64, 128, 256, 32))
        self.head = nn.Conv3d(32, 4, kernel_size=1)
    def forward(self, x): return self.head(self.backbone(x))

def get_models():
    return {
        "UNet-3D": UNet(spatial_dims=3, in_channels=4, out_channels=4, channels=(16, 32, 64, 128), strides=(2, 2, 2)).to(DEVICE),
        "DeepLabV3Plus-3D": DeepLabV3Plus3D().to(DEVICE),
        "Swin-UNETR": SwinUNETR(in_channels=4, out_channels=4, spatial_dims=3, use_checkpoint=True).to(DEVICE)
    }

In [11]:
all_p = [f.replace("_img.npy", "") for f in os.listdir(SAVE_DIR) if "_img.npy" in f]
t_ids, v_ids = train_test_split(all_p, test_size=0.2, random_state=42)
train_loader = DataLoader(BraTS3DDataset(t_ids), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(BraTS3DDataset(v_ids), batch_size=1)

In [12]:
def compute_metrics(pred, target):
    p = pred.argmax(1).flatten().cpu().numpy()
    t = target.flatten().cpu().numpy()

    iou = jaccard_score(t, p, average='macro', labels=[1,2,3], zero_division=0)
    dice = (2 * iou) / (1 + iou) if iou > 0 else 0

    t_bin, p_bin = (t > 0), (p > 0)
    tp = np.sum(t_bin & p_bin)
    tn = np.sum(~t_bin & ~p_bin)
    fp = np.sum(~t_bin & p_bin)
    fn = np.sum(t_bin & ~p_bin)

    return {
        "Dice": dice, "IoU": iou,
        "Sensitivity": tp/(tp+fn+1e-7),
        "Specificity": tn/(tn+fp+1e-7),
        "Accuracy": accuracy_score(t, p)
    }

In [13]:
def train_model(name, model, train_loader, val_loader):
    print(f"\nKhởi chạy Pipeline: {name}")
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=5, factor=0.5)
    scaler = torch.amp.GradScaler('cuda')
    criterion = DiceCELoss(softmax=True, to_onehot_y=True)

    history = []; best_dice = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

        for i, (x, y) in enumerate(pbar):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast('cuda'):
                out = model(x)
                loss = criterion(out, y.unsqueeze(1)) / ACCUM_STEPS

            scaler.scale(loss).backward()
            if (i + 1) % ACCUM_STEPS == 0:
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

            train_loss += loss.item() * ACCUM_STEPS
            pbar.set_postfix({'loss': f"{loss.item()*ACCUM_STEPS:.4f}"})

        # Giải phóng VRAM sau Train
        torch.cuda.empty_cache(); gc.collect()

        # Validation
        model.eval()
        m_list = []
        with torch.no_grad():
            for xv, yv in val_loader:
                with torch.amp.autocast('cuda'):
                    pred = model(xv.to(DEVICE))
                m_list.append(compute_metrics(pred, yv))

        avg_m = pd.DataFrame(m_list).mean().to_dict()
        avg_m.update({"epoch": epoch+1, "train_loss": train_loss/len(train_loader)})
        history.append(avg_m)

        # Lưu log & Model tốt nhất
        pd.DataFrame(history).to_csv(f"log_3D_{name}.csv", index=False)
        if avg_m['Dice'] > best_dice:
            best_dice = avg_m['Dice']
            torch.save(model.state_dict(), f"best_3D_{name}.pth")

        scheduler.step(avg_m['Dice'])
        print(f"✅ Epoch {epoch+1} | Dice: {avg_m['Dice']:.4f} | IoU: {avg_m['IoU']:.4f} | LR: {optimizer.param_groups[0]['lr']}")

In [8]:
models = get_models()
for name, model in models.items():
    train_model(name, model, train_loader, val_loader)

BasicUNet features: (32, 32, 64, 128, 256, 32).


In [14]:
train_model("Swin-UNETR", models["Swin-UNETR"], train_loader, val_loader)


Khởi chạy Pipeline: Swin-UNETR


OutOfMemoryError: CUDA out of memory. Tried to allocate 970.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 4.34 GiB is allocated by PyTorch, and 1.75 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)